### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import sys
import platform

print(sys.version)

strong_pc = platform.system() == "Linux"
in_colab = "google.colab" in sys.modules
if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    #!pip install tensorflow==2.11.0
    #!pip install tensorflow_text==2.11.0
    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules and False:
        print("Installing keras")
        !pip install keras==2.11.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0


if "DEEPNOTE_ENV" in os.environ:
    os.chdir("/..")
    os.chdir("datasets")
    os.chdir("googledrivedeepnoteintegration")
    os.chdir("Human_Data_Analytics_Project_2023")
    if not "librosa" in sys.modules:
        print("Installing Librosa")
        !pip install librosa
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

# BASE LIBRARIES
import numpy as np
import pandas as pd
import h5py
import shutil
import time
import random
import subprocess
import itertools
import warnings
import pickle
import json

# PLOT LIBRARIES
import matplotlib

# import matplotlib.pyplot as plt
# %matplotlib inline
import IPython.display as ipd

# import plotly.express as px

# AUDIO LIBRARIES
import librosa
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft, ifft, fftfreq, fftshift
from scipy.signal import stft, spectrogram, periodogram

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.utils import check_random_state
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import load_model

# GPU SETTINGS FOR LINUX and repressing warnings for windows. References for gpu: https://www.tensorflow.org/guide/gpu
show_gpu_activity = False
if sys.platform == "linux" and not in_colab:
    if show_gpu_activity:
        tf.debugging.set_log_device_placement(True)

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        # Restrict TensorFlow to only allocate a part of memory on the first GPU
        try:
            tf.config.set_logical_device_configuration(
                gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6800)]
            )
            logical_gpus = tf.config.list_logical_devices("GPU")
            print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        except RuntimeError as e:
            # Virtual devices must be set before GPUs have been initialized
            print(e)
else:
    warnings.filterwarnings("ignore", category=UserWarning)

from keras import layers
from keras import models
from keras.utils import plot_model as tf_plot

if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
# import keras_tune as kt
from keras import layers
import keras_tuner as kt
from tensorflow import keras
from keras.regularizers import L1L2
from keras.models import load_model

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# EVALUATION LIBRAIRES
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
)
from sklearn.metrics import precision_recall_fscore_support, auc

# OUR PERSONAL FUNCTIONS
import importlib
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    one_random_audio,
    plot_clip_overview,
    Spectral_Analysis,
)
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

from Preprocessing.data_loader import load_metadata
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    plot_history,
    confusion_matrix,
    listen_to_wrong_audio,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "Data", "ESC-10-depth")
samplerate = 44100

# 4 ENCODER FOR HIGH-LEVEL FEATURE EXTRACTION

In this last chapter, after training the autoencoders, we use only the best encoders as feature extractors and train the best result models from the supervised learning section on the ESC-50 encoded dataset. 

In [ ]:
import importlib

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))
importlib.reload(importlib.import_module("Preprocessing.data_loader"))
from Models.ann_utils import *
from Preprocessing.data_loader import reshape_US
from Visualization.model_plot import *

## 4.1 Classification on encoded raw audio

Since neither the basic machine learning approaches nor the dense feed forward NNs were able to achieve a good accuracy, we decided to use only the RNN approach, which at least achived alone 28% of accurcy on the 10 class classification problem from raw audio.

### Create the dataset

In [ ]:
batch_size = 30 if not strong_pc else 128

dataset, label = create_dataset_lite(df_ESC10, batch_size=batch_size, ndim=2)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=0)

### Load the encoder and build the model

In [ ]:
AE_name = "Dense_AE_ffnn"
path_to_AE = os.path.join(main_dir, "Saved_Models", AE_name)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "encoder_classifier_raw"

In [ ]:
def build_model(
    encoder=encoder,
    n_labels=n_labels,
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
    name=Classifier_name,
):

    classifier = tf.keras.Sequential(
        [
            tf.keras.layers.GRU(units=n_units, activation=activation),
            tf.keras.layers.Dense(n_labels, activation="softmax"),
        ],
        name="Classifier",
    )

    # build the model with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    # add 1 dimension to the code
    code = tf.expand_dims(code, axis=2)
    output = classifier(code)
    model = tf.keras.Model(inputs=inp, outputs=output, name=Classifier_name)

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )

    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    return model

In [ ]:
model = build_model()
model.summary()

### Run a Grid Search to find the best hyperparameters

In [ ]:
# params = {'learning_rate':[1e-3, 1e-4]}
params = {
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "n_units": [8, 32, 128],
    "activation": ["relu", "elu", "tanh"],
}
epochs = 1 if not strong_pc else 100
patience = 10
verbose = 0
K_fold = 5
model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=verbose,
    K=K_fold,
)

### Train the model

In [ ]:
seed = 42
tf.random.set_seed(seed)

path_to_ESC10 = os.path.join(main_dir, "Data", "ESC-10-depth")

# create the dataset
batch_size = 80
preprocessing = None
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
)

# build the model with the best parameters
best_params = {"activation": "elu", "learning_rate": 0.01, "n_units": 8}
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 100
patience = 10
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=0,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

## 4.2 Classification on encoded spectrograms - RNN

### Short Time Fourier Transform

#### Create dataset

In [ ]:
batch_size = 30
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoders and build the model

We trained 2 AEs to produce a flatten code. On these code we are going to train a RNN classifier.

In [ ]:
AE_STFT = "AE_Conv_prep_flatten_STFT"

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "encoder_classifier_flat_code_STFT"

In [ ]:
def build_model(
    encoder=encoder,
    INPUT_DIM=INPUT_DIM,
    n_labels=n_labels,
    n_units=8,
    activation="tanh",
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    classifier = tf.keras.Sequential(name="classifier")
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            classifier.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    classifier.add(tf.keras.layers.Flatten())
    classifier.add(
        tf.keras.layers.Dense(n_labels, activation="softmax", name="Final_dense")
    )

    # build the model with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    # add 1 dimension to the code
    code = tf.expand_dims(code, axis=1)
    output = classifier(code)
    model = tf.keras.Model(inputs=inp, outputs=output, name=Classifier_name)

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )

    if compile:
        # compile the model
        model.compile(loss=loss, optimizer=optimizer, metrics=metrics)
        if verbose > 0:
            display(model.summary())

    return model

In [ ]:
# model = build_model(encoder = encoder_list[0] , n_hidden_layers = 5 )
model = build_model(n_hidden_layers=5)

display(model.summary(line_length=100))
display(model.layers[1].summary(line_length=100))
display(model.layers[3].summary(line_length=100))

#### Run a Grid Search to find the best hyperparameters

In [ ]:
# params = {'n_hidden_layers':[2,4]}
params = {
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "n_units": [8, 32, 128],
    "activation": ["relu", "elu", "tanh"],
    "n_hidden_layers": [1, 2, 3, 4],
}
epochs = 1 if not strong_pc else 50
patience = 10
verbose = 0
K_fold = 5

model_cv, result, best_params = K_fold_training(
    dataset_STFT,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=verbose,
    K=K_fold,
)

#### Train the model

In [ ]:
tf.random.set_seed(seed)

path_to_ESC50 = os.path.join(main_dir, "Data", "ESC-50-depth")

# create the dataset
batch_size = 30
preprocessing = "STFT"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC50,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=3,
    resize=True,
)

In [ ]:
# build the model with the best parameters
best_params = {
    "activation": "tanh",
    "learning_rate": 0.0001,
    "n_hidden_layers": 4,
    "n_units": 128,
}
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 50
patience = 20
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=False,
)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### Mel Frequency Cepstral Coefficients

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "MFCC"
n_dim = 3

dataset_MFCC, label_MFCC = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_MFCC, label_names=list(label_MFCC.columns), verbose=0, show_figure=True
)

#### Load the encoder and build the model

In [ ]:
AE_MFCC = "AE_Conv_prep_flatten_MFCC"

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_MFCC)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "encoder_classifier_flat_code_MFCC"

In [ ]:
INPUT_DIM = (64, 128, 1)


def build_model(
    encoder=encoder,
    INPUT_DIM=INPUT_DIM,
    n_labels=n_labels,
    n_units=8,
    activation="tanh",
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    classifier = tf.keras.Sequential(name="classifier")
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            classifier.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    classifier.add(tf.keras.layers.Flatten())
    classifier.add(
        tf.keras.layers.Dense(n_labels, activation="softmax", name="Final_dense")
    )

    # build the model with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    # add 1 dimension to the code
    code = tf.expand_dims(code, axis=1)
    output = classifier(code)
    model = tf.keras.Model(inputs=inp, outputs=output, name=Classifier_name)

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )

    if compile:
        # compile the model
        model.compile(loss=loss, optimizer=optimizer, metrics=metrics)
        if verbose > 0:
            display(model.summary())

    return model

In [ ]:
# model = build_model(encoder = encoder_list[0] , n_hidden_layers = 5 )
model = build_model(n_hidden_layers=5)

display(model.summary(line_length=100))
display(model.layers[1].summary(line_length=100))
display(model.layers[3].summary(line_length=100))

#### Run a grid search to find the best params

In [ ]:
# params = {'n_hidden_layers':[2,4]}
params = {
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "n_units": [8, 32, 128],
    "activation": ["relu", "elu", "tanh"],
    "n_hidden_layers": [1, 2, 3, 4],
}
epochs = 1 if not strong_pc else 50
patience = 10
verbose = 0
K_fold = 5

model_cv, result, best_params = K_fold_training(
    dataset_MFCC,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=verbose,
    K=K_fold,
)

#### Train the model

In [ ]:
seed = 42
tf.random.set_seed(seed)

path_to_ESC50 = os.path.join(main_dir, "Data", "ESC-50-depth")

# create the dataset
batch_size = 30
preprocessing = "MFCC"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC50,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=3,
    resize=True,
)

In [ ]:
# build the model with the best parameters
best_params = {
    "activation": "elu",
    "learning_rate": 0.001,
    "n_hidden_layers": 4,
    "n_units": 128,
}
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 50
patience = 20
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=False,
)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

## 4.3 Classification on encoded spectrograms - CNN

Two out of five autoencoders produce a multichannel code. We kept the number of channels small in order to plot the latent space and avoid having a too big latent space.  

### Short Time Fourier Transform

#### Create dataset

In [ ]:
batch_size = 30 if not strong_pc else 128
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoders and build the model

In [ ]:
AE_STFT = "Fully_Convolutional_AE_STFT"

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = tf.keras.models.load_model(path_to_AE, compile=False)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "encoder_classifier_flat_code_STFT"

In [ ]:
INPUT_DIM = (64, 128, 1)


def build_model(
    encoder=encoder,
    INPUT_DIM=INPUT_DIM,
    n_labels=n_labels,
    activation="tanh",
    n_layers=2,
    n_units=4,
    kernel_size=(3, 3),
    strides=(2, 2),
    max_pooling=(1, 1),
    batch_norm=True,
    learning_rate=1e-3,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    classifier = tf.keras.Sequential(name="Classifier")
    for i in range(n_layers):
        filters = n_units * (i + 1)
        name = "Conv_layer_" + str(i + 1)
        classifier.add(
            tf.keras.layers.Conv2D(
                filters,
                kernel_size,
                strides=2,
                activation=activation,
                padding="same",
                name=name,
            )
        )
        name = "Max_pooling_layer_" + str(i + 1)
        classifier.add(layers.MaxPool2D(max_pooling, padding="same", name=name))

        if batch_norm:
            name = "Batch_Norm_layer_" + str(i + 1)
            classifier.add(layers.BatchNormalization(name=name))

    # Flatten the output of the previous layer
    classifier.add(tf.keras.layers.Flatten(name="Flatten_layer"))

    # Dense layer for classification
    classifier.add(
        tf.keras.layers.Dense(n_labels, activation="softmax", name="Final_dense")
    )

    # build the model with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    # add 1 dimension to the code
    output = classifier(code)
    model = tf.keras.Model(inputs=inp, outputs=output, name=Classifier_name)

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )

    if compile:
        # compile the model
        model.compile(loss=loss, optimizer=optimizer, metrics=metrics)
        if verbose > 0:
            display(model.summary())

    return model

In [ ]:
# model = build_model(encoder = encoder_list[0] , n_hidden_layers = 5 )
model = build_model(kernel_size=(8, 8))

display(model.summary(line_length=100))
display(model.layers[1].summary(line_length=100))
display(model.layers[2].summary(line_length=100))

In [ ]:
tf.keras.utils.plot_model(
    model,
    expand_nested=True,
    show_shapes=True,
    dpi=100,  # size of the image
)

#### Run a Grid Search to find the best hyperparameters

In [ ]:
# params = {'n_layers':[2,4]}
params = {
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "n_units": [8, 64],
    "activation": ["relu", "elu", "tanh"],
    "n_layers": [1, 2, 3, 4],
    "kernel_size": [(2, 2), (3, 3), (4, 4)],
    "strides": [(1, 1), (2, 2)],
}

epochs = 1 if not strong_pc else 50
patience = 10
verbose = 0
K_fold = 5

model_cv, result, best_params = K_fold_training(
    dataset_STFT,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=verbose,
    K=K_fold,
)

# 2160 fits with less then 10 seconds each on CPU ---> 5 hours on GPU?

#### Train the model

In [ ]:
tf.random.set_seed(seed)

path_to_ESC50 = os.path.join(main_dir, "Data", "ESC-50-depth")

# create the dataset
batch_size = 30
preprocessing = "STFT"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC50,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=3,
    resize=True,
    transpose=False,
)

In [ ]:
# build the model with the best parameters
best_params = {
    "activation": "elu",
    "kernel_size": (2, 2),
    "learning_rate": 0.0001,
    "n_layers": 2,
    "n_units": 64,
    "strides": (2, 2),
}
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 50
patience = 20
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=False,
)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### Mel Frequency Cepstral Coefficients

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "MFCC"
n_dim = 3

dataset_MFCC, label_MFCC = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_MFCC, label_names=list(label_MFCC.columns), verbose=0, show_figure=True
)

#### Load the encoder and build the model

In [ ]:
AE_MFCC = "Fully_Convolutional_AE_MFCC"

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_MFCC)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "encoder_classifier_flat_code_MFCC"

In [ ]:
def build_model(
    encoder=encoder,
    INPUT_DIM=INPUT_DIM,
    n_labels=n_labels,
    activation="tanh",
    n_layers=2,
    n_units=4,
    kernel_size=(3, 3),
    strides=(2, 2),
    max_pooling=(1, 1),
    batch_norm=True,
    learning_rate=1e-3,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    classifier = tf.keras.Sequential(name="Classifier")
    for i in range(n_layers):
        filters = n_units * (i + 1)
        name = "Conv_layer_" + str(i + 1)
        classifier.add(
            tf.keras.layers.Conv2D(
                filters,
                kernel_size,
                strides=2,
                activation=activation,
                padding="same",
                name=name,
            )
        )
        name = "Max_pooling_layer_" + str(i + 1)
        classifier.add(layers.MaxPool2D(max_pooling, padding="same", name=name))

        if batch_norm:
            name = "Batch_Norm_layer_" + str(i + 1)
            classifier.add(layers.BatchNormalization(name=name))

    # Flatten the output of the previous layer
    classifier.add(tf.keras.layers.Flatten(name="Flatten_layer"))

    # Dense layer for classification
    classifier.add(
        tf.keras.layers.Dense(n_labels, activation="softmax", name="Final_dense")
    )

    # build the model with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    # add 1 dimension to the code
    output = classifier(code)
    model = tf.keras.Model(inputs=inp, outputs=output, name=Classifier_name)

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )

    if compile:
        # compile the model
        model.compile(loss=loss, optimizer=optimizer, metrics=metrics)
        if verbose > 0:
            display(model.summary())

    return model

In [ ]:
# model = build_model(encoder = encoder_list[0] , n_hidden_layers = 5 )
model = build_model(n_layers=4)

display(model.summary(line_length=100))
display(model.layers[1].summary(line_length=100))
display(model.layers[2].summary(line_length=100))

In [ ]:
tf.keras.utils.plot_model(
    model,
    expand_nested=True,
    show_shapes=True,
    dpi=100,  # size of the image
)

#### Run a grid search to find the best params

In [ ]:
# params = {'n_layers':[2,4]}
params = {
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "n_units": [8, 64],
    "activation": ["relu", "elu", "tanh"],
    "n_layers": [1, 2, 3, 4],
    "kernel_size": [(2, 2), (3, 3), (4, 4)],
    "strides": [(1, 1), (2, 2)],
}

epochs = 1 if not strong_pc else 50
patience = 10
verbose = 0
K_fold = 5

model_cv, result, best_params = K_fold_training(
    dataset_MFCC,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=verbose,
    K=K_fold,
)

#### Train the model

In [ ]:
seed = 42
tf.random.set_seed(seed)

path_to_ESC50 = os.path.join(main_dir, "Data", "ESC-50-depth")

# create the dataset
batch_size = 30
preprocessing = "MFCC"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC50,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=3,
    resize=True,
)

In [ ]:
# build the model with the best parameters
best_params = {
    "activation": "relu",
    "kernel_size": (4, 4),
    "learning_rate": 0.0001,
    "n_layers": 2,
    "n_units": 64,
    "strides": (1, 1),
}
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 50
patience = 20
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=False,
)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

## 4.4 Classification on encoded spectrograms - SVM

As a last attempt, we decided to train an SVM always on the ESC-50 encoded in a latent flatten space.

### STFT input, flatten code

#### Create the dataset

In [ ]:
batch_size = 30 if not strong_pc else 128
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC50,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoder

In [ ]:
AE_STFT = "AE_Conv_prep_flatten_STFT"
INPUT_DIM = (64, 128, 1)

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "SVM_encoder_classifier_flat_code_STFT"

#### Run a grid search to find the best params

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [ ]:
# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# Define the SVM model
svm = SVC()

# Define the hyperparameter grid
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ["rbf", "poly", "sigmoid"],
    "degree": [2, 3, 4, 5],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1, 10],
}

# Define the k-fold cross-validation strategy (e.g., 5-fold cross-validation)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create the GridSearchCV object with cross-validation
grid_search = GridSearchCV(
    estimator=svm, param_grid=param_grid, cv=kfold, scoring="accuracy"
)

# Fit the GridSearchCV object to your data
grid_search.fit(X, y)

# Access the best hyperparameters and the best SVM model
best_params = grid_search.best_params_
best_svm = grid_search.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters:", best_params)

# Evaluate the best SVM model using cross-validation or a separate test set
cv_results = cross_val_score(best_svm, X, y, cv=kfold, scoring="accuracy")
print(
    "Cross-Validation Accuracy: {:.2f}% (+/- {:.2f}%)".format(
        cv_results.mean() * 100, cv_results.std() * 100
    )
)

In [ ]:
# show only the best model

# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# params
C = 0.001
kernel = "poly"
gamma = 0.01
degree = 2
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=123
)

# fit the model
start_time = time.time()
model = SVC(C=C, kernel=kernel, random_state=123, degree=degree, gamma=gamma)
model.fit(X_train, y_train)

# accuracy on train and test
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
print(
    "Accuracy on train: {:.2f}%".format(accuracy_score(y_train, y_predict_train) * 100)
)
print("Accuracy on test: {:.2f}%".format(accuracy_score(y_test, y_predict_test) * 100))

# confusion matrix
labels50 = list(label_STFT.columns)
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### MFCC input, flatten code

#### Create the dataset

In [ ]:
batch_size = 30 if not strong_pc else 128
preprocessing = "MFCC"
n_dim = 3

dataset_MFCC, label_MFCC = create_dataset_lite(
    df_ESC50,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_MFCC, label_names=list(label_MFCC.columns), verbose=0, show_figure=True
)

#### Load the encoder

In [ ]:
AE_MFCC = "AE_Conv_prep_flatten_MFCC"
INPUT_DIM = (64, 128, 1)

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_MFCC)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "SVM_encoder_classifier_flat_code_MFCC"

#### Run a grid search to find the best params

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [ ]:
# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_MFCC.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_MFCC.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# Define the SVM model
svm = SVC()

# Define the hyperparameter grid
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ["rbf", "poly", "sigmoid"],
    "degree": [2, 3, 4, 5],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1, 10],
}

# Define the k-fold cross-validation strategy (e.g., 5-fold cross-validation)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create the GridSearchCV object with cross-validation
grid_search = GridSearchCV(
    estimator=svm, param_grid=param_grid, cv=kfold, scoring="accuracy"
)

# Fit the GridSearchCV object to your data
grid_search.fit(X, y)

# Access the best hyperparameters and the best SVM model
best_params = grid_search.best_params_
best_svm = grid_search.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters:", best_params)

# Evaluate the best SVM model using cross-validation or a separate test set
cv_results = cross_val_score(best_svm, X, y, cv=kfold, scoring="accuracy")
print(
    "Cross-Validation Accuracy: {:.2f}% (+/- {:.2f}%)".format(
        cv_results.mean() * 100, cv_results.std() * 100
    )
)

In [ ]:
# show only the best model

# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# params
C = 0.001
kernel = "poly"
gamma = 10
degree = 3
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=123
)

# fit the model
start_time = time.time()
model = SVC(C=C, kernel=kernel, random_state=123, degree=degree, gamma=gamma)
model.fit(X_train, y_train)

# accuracy on train and test
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
print(
    "Accuracy on train: {:.2f}%".format(accuracy_score(y_train, y_predict_train) * 100)
)
print("Accuracy on test: {:.2f}%".format(accuracy_score(y_test, y_predict_test) * 100))

# confusion matrix
labels50 = list(label_STFT.columns)
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### STFT input, flatten code, SSIM loss for the AE

#### Create the dataset

In [ ]:
batch_size = 30 if not strong_pc else 128
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC50,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoder

In [ ]:
from tensorflow.image import ssim


def ssim_loss(y_true, y_pred):
    return 1 - tf.reduce_mean(ssim(y_true, y_pred, max_val=1.0))

In [ ]:
AE_STFT = "AE_Conv_prep_flatten_STFT_code_size_32_ssim"
INPUT_DIM = (64, 128, 1)

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = tf.keras.models.load_model(
    os.path.join(main_dir, "Saved_Models", AE_STFT),
    custom_objects={"ssim_loss": ssim_loss},
)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "SVM_encoder_classifier_flat_code_STFT_ssim"

#### Run a grid search to find the best params

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [ ]:
# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# Define the SVM model
svm = SVC()

# Define the hyperparameter grid
param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ["rbf", "poly", "sigmoid"],
    "degree": [2, 3, 4, 5],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1, 10],
}

# Define the k-fold cross-validation strategy (e.g., 5-fold cross-validation)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create the GridSearchCV object with cross-validation
grid_search = GridSearchCV(
    estimator=svm, param_grid=param_grid, cv=kfold, scoring="accuracy"
)

# Fit the GridSearchCV object to your data
grid_search.fit(X, y)

# Access the best hyperparameters and the best SVM model
best_params = grid_search.best_params_
best_svm = grid_search.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters:", best_params)

# Evaluate the best SVM model using cross-validation or a separate test set
cv_results = cross_val_score(best_svm, X, y, cv=kfold, scoring="accuracy")
print(
    "Cross-Validation Accuracy: {:.2f}% (+/- {:.2f}%)".format(
        cv_results.mean() * 100, cv_results.std() * 100
    )
)

In [ ]:
# show only the best model

# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# params
C = 10
kernel = "poly"
gammma = "scale"
degree = 5
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=123
)

# fit the model
start_time = time.time()
model = SVC(C=C, kernel=kernel, random_state=123, degree=degree, gamma=gamma)
model.fit(X_train, y_train)

# accuracy on train and test
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
print(
    "Accuracy on train: {:.2f}%".format(accuracy_score(y_train, y_predict_train) * 100)
)
print("Accuracy on test: {:.2f}%".format(accuracy_score(y_test, y_predict_test) * 100))

# confusion matrix
labels50 = list(label_STFT.columns)
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### STFT input, flatten code, Autoencoder trained on AudioSet database - classification on ESC-50

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC50,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoder

In [ ]:
AE_STFT = "AE_Conv_prep_flatten_STFT_AudioSet"
INPUT_DIM = (64, 128, 1)

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "SVM_encoder_classifier_flat_code_STFT_AudioSet"

#### Run a grid search to find the best params

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [ ]:
# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# Define the SVM model
svm = SVC()

# Define the hyperparameter grid
"""param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'kernel': ['rbf', 'sigmoid', 'poly'],
    'degree': [2,3],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1, 10]
}"""

param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1, 10],
}

# Define the k-fold cross-validation strategy (e.g., 5-fold cross-validation)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create the GridSearchCV object with cross-validation
grid_search = GridSearchCV(
    estimator=svm, param_grid=param_grid, cv=kfold, scoring="accuracy", verbose=2
)

# Fit the GridSearchCV object to your data
grid_search.fit(X, y)

# Access the best hyperparameters and the best SVM model
best_params = grid_search.best_params_
best_svm = grid_search.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters:", best_params)

# Evaluate the best SVM model using cross-validation or a separate test set
cv_results = cross_val_score(best_svm, X, y, cv=kfold, scoring="accuracy")
print(
    "Cross-Validation Accuracy: {:.2f}% (+/- {:.2f}%)".format(
        cv_results.mean() * 100, cv_results.std() * 100
    )
)

In [ ]:
# show only the best model

# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# params
C = 100
kernel = "rbf"
gamma = "scale"
# degree = 2
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=123
)

# fit the model
start_time = time.time()
model = SVC(C=C, kernel=kernel, random_state=123, gamma=gamma)  # , degree = degree)
model.fit(X_train, y_train)

# accuracy on train and test
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
print(
    "Accuracy on train: {:.2f}%".format(accuracy_score(y_train, y_predict_train) * 100)
)
print("Accuracy on test: {:.2f}%".format(accuracy_score(y_test, y_predict_test) * 100))

# confusion matrix
labels50 = list(label_STFT.columns)
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)

### STFT input, flatten code, Autoencoder trained on ESC-50 Augmented database - classification on the original ESC-50 

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "STFT"
n_dim = 3

dataset_STFT, label_STFT = create_dataset_lite(
    df_ESC50,
    batch_size=batch_size,
    preprocessing=preprocessing,
    ndim=n_dim,
    resize=True,
)

INPUT_DIM, n_labels = example_batch(
    dataset_STFT, label_names=list(label_STFT.columns), verbose=0, show_figure=True
)

#### Load the encoder

In [ ]:
AE_STFT = "AE_Conv_prep_flatten_STFT_Augmented"
INPUT_DIM = (64, 128, 1)

path_to_AE = os.path.join(main_dir, "Saved_Models", AE_STFT)

# import the autencoder
autoencoder = load_model(path_to_AE)
encoder = autoencoder.layers[1]

# freeze the encoder
encoder.trainable = False
encoder.summary()

Classifier_name = "SVM_encoder_classifier_flat_code_STFT_AE_trained_on_Augmented"

#### Run a grid search to find the best params

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

Runniamo la grid search sul ESC-50 originale

In [ ]:
# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# Define the SVM model
svm = SVC()

# Define the hyperparameter grid
"""param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'kernel': ['rbf', 'sigmoid', 'poly'],
    'degree': [2,3],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1, 10]
}"""

param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1, 10],
}

# Define the k-fold cross-validation strategy (e.g., 5-fold cross-validation)
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Create the GridSearchCV object with cross-validation
grid_search = GridSearchCV(
    estimator=svm, param_grid=param_grid, cv=kfold, scoring="accuracy", verbose=2
)

# Fit the GridSearchCV object to your data
grid_search.fit(X, y)

# Access the best hyperparameters and the best SVM model
best_params = grid_search.best_params_
best_svm = grid_search.best_estimator_

# Print the best hyperparameters
print("Best Hyperparameters:", best_params)

# Evaluate the best SVM model using cross-validation or a separate test set
cv_results = cross_val_score(best_svm, X, y, cv=kfold, scoring="accuracy")
print(
    "Cross-Validation Accuracy: {:.2f}% (+/- {:.2f}%)".format(
        cv_results.mean() * 100, cv_results.std() * 100
    )
)

#### Best model evaluation

In [ ]:
# show only the best model

# Create the dataset
X = np.array(
    [encoder(np.expand_dims(x, axis=0)) for x, y in dataset_STFT.unbatch().take(2000)]
)
X = np.squeeze(X, axis=1)
y = np.array([y for x, y in dataset_STFT.unbatch().take(2000)])
y = np.argmax(y, axis=1)

# params
C = 10
kernel = "rbf"
gamma = 0.01
# degree = 2
test_size = 0.25
name_df = "ESC50"

# train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=123
)

# fit the model
start_time = time.time()
model = SVC(C=C, kernel=kernel, random_state=123, gamma=gamma)  # , degree = degree)
model.fit(X_train, y_train)

# accuracy on train and test
y_predict_train = model.predict(X_train)
y_predict_test = model.predict(X_test)
print(
    "Accuracy on train: {:.2f}%".format(accuracy_score(y_train, y_predict_train) * 100)
)
print("Accuracy on test: {:.2f}%".format(accuracy_score(y_test, y_predict_test) * 100))

# confusion matrix
labels50 = list(label_STFT.columns)
confusion_mtx = confusion_matrix(y_test, y_predict_test, labels50)

# listen to wrong audio
listen_to_wrong_audio(
    df_ESC50, y_test, y_predict_test, confusion_mtx=confusion_mtx, labels=labels50
)  # , n_audio = 2)

In [ ]:
# model path
model_path = os.path.join(main_dir, "Saved_Models", Classifier_name)

# save the model
with open(model_path, "wb") as f:
    pickle.dump(model, f)